In [2]:
import pandas as pd

In [ ]:
# ## ----- the directory convention used is based off the google drive organization ------ ##

In [20]:
# rheo_df = pd.read_excel("/Users/mauriellenoto/Desktop/seager/ionicliquids/isn/UAFW/UAFW_20260707_rheo.xls")

import pandas as pd
tga_df = pd.read_excel('/Users/mauriellenoto/Desktop/seager/ionicliquids/isn/TGA/2026-07-13/2026-07-13-TGA-UAFW.xls',
                       sheet_name=None)
tga_df = pd.read_excel('/Users/mauriellenoto/Desktop/seager/ionicliquids/isn/TGA/2026-07-13/2026-07-13-TGA-UAFW.xls',
                       sheet_name="Ramp 10.00 °Cmin to 650.00 °C", header=1)
cols = list(tga_df.columns)
tga_df
# print(type(cols))
# sheets = list(tga_df.keys())

# data = sheets[1:]
# sheets


,Time,Temperature,Weight,Weight.1,Deriv. Weight
0,min,°C,mg,%,% / °C
1,0.01,22.81,7.316,100.077,-0.347624
2,0.02,22.81,7.325,100.2,-0.347624
3,0.04,22.81,7.327,100.224,-0.347624
4,0.06,22.81,7.327,100.23,-0.347624
...,...,...,...,...,...
3773,62.87,647.12,0.002,0.031,0.001184
3774,62.89,647.29,0.002,0.031,0.001186
3775,62.91,647.45,0.002,0.032,0.001187
3776,62.92,647.62,0.002,0.032,0.001189


In [ ]:

# if you want a df of the details, use:
# details_df = pd.read_excel("path_to_excel_file", sheet_name = "Details", header = 1)

def extract_data(path): 
    # Read in original file 
    xl = pd.ExcelFile(path) 
    all_sheets = xl.sheet_names 
    print(all_sheets) 
    
    data_sheets = all_sheets[1:] 
    df_list = [] 
    
    for sheet in data_sheets: 
        sheet_df = pd.read_excel(path, sheet_name=sheet, header=1) 
        
        # extract the units from the first row of data (index 0)
        units = sheet_df.iloc[0].fillna('').astype(str).tolist()
        
        # combine old column names with the units
        new_columns = []
        for col, unit in zip(sheet_df.columns, units):
            clean_col = col.split('.')[0] if '.' in col else col
            if unit:
                new_columns.append(f"{clean_col} ({unit})")
            else:
                new_columns.append(clean_col)
                
        # assign the new combined names back to the dataframe columns
        sheet_df.columns = new_columns
        
        # drop units row from the data
        sheet_df = sheet_df.drop(index=0).reset_index(drop=True)
        
        # add source sheet column
        sheet_df['Source Sheet'] = sheet 
        df_list.append(sheet_df) 
        
    # Vertically stack all the sheets together into one final df 
    combined_df = pd.concat(df_list, ignore_index=True)
    
    # convert to float
    # errors='coerce' turns any unconvertible text or bad data into NaN safely
    for col in combined_df.columns:
        if col != 'Source Sheet':
            combined_df[col] = pd.to_numeric(combined_df[col], errors='coerce')
    
    # make sure source sheet is last column in final df
    cols = [col for col in combined_df.columns if col != 'Source Sheet'] + ['Source Sheet']
    combined_df = combined_df[cols]
    
    return combined_df
